# Denmark: BikeNodePlanner–OSM overlap and surfaces

Nationwide version of `one_region_analysis.ipynb`, fixed to `SAMPLE = "denmark"`.

The BNN input is `bikenwloops/data/input/denmark/network/processed/edges.gpkg`. No slope or nodes file is required. Plotting node markers are derived from unique edge endpoints; they are not loaded from a separate node dataset. Original `edge_id` values are retained; `link_id` identifies each input row for surface segmentation and imputation.

The notebook reuses the existing nationwide OSM cache (`outputs/denmark/denmark_bike_edges.parquet`) and writes its reports, plots and background cache under `outputs/denmark/`. Run all cells in order from this folder or the repository root. Nationwide matching and plotting take longer than individual regions.


## Setup

In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely
from IPython.display import display
from shapely.geometry import box
import requests

working_dir = Path.cwd().resolve()

PROJECT_ROOT = working_dir.parent if working_dir.name == "osm_data_exploration" else working_dir

DENMARK_OSM_DIR = PROJECT_ROOT / "osm_data_exploration/outputs/denmark"
PBF_PATH = DENMARK_OSM_DIR / "denmark-latest.osm.pbf"

DENMARK_OSM_DIR.mkdir(parents=True, exist_ok=True)

EXTRACTION_MARGIN_M = 5_000
TOLERANCE_M = 10

This notebook always analyses Denmark. Configure the matching tolerance in the setup cell above. Surface groups, colours, matching and imputation follow the single-region notebook.


In [ ]:
SAMPLE = "denmark"

BIKE_LOOPS_DIR = PROJECT_ROOT / "bikenwloops/data/input" / SAMPLE / "network/processed"
BNN_EDGES_PATH = BIKE_LOOPS_DIR / "edges.gpkg"
OUTPUT_DIR = PROJECT_ROOT / "osm_data_exploration/outputs" / SAMPLE
# Reuse the existing national cache instead of creating a duplicate national file.
OSM_EDGES_PATH = OUTPUT_DIR / "denmark_bike_edges.parquet"

if not BNN_EDGES_PATH.is_file():
    raise FileNotFoundError(f"Missing Denmark BNN edges: {BNN_EDGES_PATH}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Selected region:", SAMPLE)
print("BNN edges:", BNN_EDGES_PATH)
print("OSM cache:", OSM_EDGES_PATH)
print("Output folder:", OUTPUT_DIR)

## Load OSM Bike network

Tag information for Denmark: [LINK](https://taginfo.geofabrik.de/europe:denmark/)

In [ ]:
PBF_URL = ("https://download.geofabrik.de/europe/denmark-latest.osm.pbf")

if not PBF_PATH.exists():
    print("Downloading Denmark OSM data...")
    print("This can take several minutes.")

    temporary_path = Path(str(PBF_PATH) + ".download")

    with requests.get(PBF_URL, stream=True, timeout=120) as response:
        response.raise_for_status()
        total_bytes = int(response.headers.get("content-length", 0))
        downloaded_bytes = 0

        with temporary_path.open("wb") as file:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if not chunk:
                    continue

                file.write(chunk)
                downloaded_bytes += len(chunk)

                if total_bytes:
                    percentage = (100 * downloaded_bytes / total_bytes)
                    print(f"\rDownloaded: {percentage:5.1f}%", end="")

    temporary_path.replace(PBF_PATH)
    print("\nDownload completed.")

else:
    print("Using existing Denmark OSM file.")

print("OSM file:", PBF_PATH)

In [ ]:
# Denmark has only an edges file. Endpoint markers are derived for plotting.
bike_loops_edges = gpd.read_file(BNN_EDGES_PATH)
bike_loops_edges = bike_loops_edges.loc[
    bike_loops_edges.geometry.notna() & ~bike_loops_edges.geometry.is_empty
].copy()

if bike_loops_edges.empty:
    raise ValueError("The Denmark BNN edges file has no usable geometries.")
if bike_loops_edges.crs is None:
    raise ValueError("The Denmark BNN edges do not have a CRS.")
# All distance settings and length summaries use metres.
bike_loops_edges = bike_loops_edges.to_crs("EPSG:25832")
if not bike_loops_edges.geom_type.eq("LineString").all():
    raise ValueError("Expected LineString geometries in Denmark edges.gpkg.")
BIKE_LOOPS_CRS = bike_loops_edges.crs

endpoint_geometries = np.concatenate([
    shapely.get_point(bike_loops_edges.geometry.array, 0),
    shapely.get_point(bike_loops_edges.geometry.array, -1),
])
endpoint_geometries = gpd.GeoSeries(endpoint_geometries, crs=BIKE_LOOPS_CRS).drop_duplicates().reset_index(drop=True)
bike_loops_nodes = gpd.GeoDataFrame(geometry=endpoint_geometries, crs=BIKE_LOOPS_CRS)

print("Bike loops CRS:", BIKE_LOOPS_CRS)
print(f"BNN links: {len(bike_loops_edges):,}")
print(f"Unique endpoint markers (derived from edges): {len(bike_loops_nodes):,}")


In [ ]:
minx, miny, maxx, maxy = bike_loops_edges.total_bounds

bike_loops_bbox = box(
    minx - EXTRACTION_MARGIN_M,
    miny - EXTRACTION_MARGIN_M,
    maxx + EXTRACTION_MARGIN_M,
    maxy + EXTRACTION_MARGIN_M,
)

# Use the selected loop network coordinate system for distances in metres.
# EXTRACTION_MARGIN_M controls the extraction margin; TOLERANCE_M controls matching.
bike_loops_bbox_gdf = gpd.GeoDataFrame(
    geometry=[bike_loops_bbox],
    crs=BIKE_LOOPS_CRS,
)

# WGS84 means the bounding box coordinates have been converted to the standard global longitude/latitude coordinate system, EPSG:4326.
bike_loops_bbox_wgs84 = (
    bike_loops_bbox_gdf
    .to_crs("EPSG:4326")
    .geometry
    .iloc[0]
)

### Load plotting functions


In [ ]:
# Load plotting functions from the Python file beside this notebook.
import sys
plotting_module_dir = PROJECT_ROOT / "osm_data_exploration"
if str(plotting_module_dir) not in sys.path:
    sys.path.insert(0, str(plotting_module_dir))

from network_plotting import (
    plot_bicycle_networks,
    plot_bicycle_networks_with_export,
    plot_overlap_results,
    plot_matched_attribute,
)

### Define surface groups

In [ ]:
def group_lit(value):
    """Group OSM lit values into: missing, no or yes."""

    if pd.isna(value):
        return "missing"

    value = str(value).strip().lower()

    if value in {"yes", "automatic", "y"}:
        return "yes"

    if value == "no":
        return "no"
    
    return "missing"


# Project-specific grouping: preserve the existing paved/unpaved choices.
# Ambiguous tags remain "other" rather than being treated as missing data.
SURFACE_GROUPS = {
    # Paved
    'asphalt': 'paved', 'chipseal': 'paved', 'concrete': 'paved', 'concrete:lanes': 'paved', 'paved': 'paved', 
    'plastic': 'paved', 'rubber': 'paved', 'rubbercrumb': 'paved', 'tartan': 'paved',
    # Unpaved
    'acrylic': 'unpaved', 'artificial_turf': 'unpaved', 'bricks': 'unpaved', 'clay': 'unpaved', 
    'cobblestone': 'unpaved', 'compacted': 'unpaved', 'composite': 'unpaved', 'concrete:plates': 'unpaved', 
    'dirr': 'unpaved', 'dirt': 'unpaved', 'dirt/sand': 'unpaved', 'earth': 'unpaved', 'fine_gravel': 'unpaved',
    'grass': 'unpaved', 'grass_paver': 'unpaved', 'grass_paver;sand': 'unpaved', 'grave': 'unpaved', 'gravel': 'unpaved', 
    'ground': 'unpaved', 'grus': 'unpaved', 'metal': 'unpaved', 'metal_grate': 'unpaved', 'metal_grid': 'unpaved', 
    'mud': 'unpaved', 'non-sealed_chipseal': 'unpaved', 'paving_stones': 'unpaved', 'pebblestone': 'unpaved', 
    'rock': 'unpaved', 'rocks': 'unpaved', 'sand': 'unpaved', 'sand,_kies': 'unpaved', 'sand/gravel/stones': 'unpaved', 
    'scree': 'unpaved', 'sett': 'unpaved', 'steel': 'unpaved', 'stepping_stones': 'unpaved', 'stone': 'unpaved',
    'tiles': 'unpaved', 'unhewn_cobblestone': 'unpaved', 'unpaved': 'unpaved', 'wood': 'unpaved', 'woodchips': 'unpaved', 
    # Ambiguous / Other
    'com': 'other', 'driving_plates': 'other', 'mixed': 'other', 'pakket': 'other', 'sas': 'other', 'verdichtetw': 'other',
    # Missing / Unmatched
    "missing surface tag": "missing", "missing": "missing", "no OSM match": "unmatched",
}

def group_surface(value):
    """Use explicit project groups; reject new tags until they are reviewed."""
    if pd.isna(value) or not str(value).strip():
        return "missing"
    value = str(value).strip().lower()
    if value not in SURFACE_GROUPS:
        raise ValueError(f"Surface {value!r} needs an explicit group and map colour.")
    return SURFACE_GROUPS[value]

# Stable raw-tag colours shared by original and imputed interactive maps.
surface_palette = {
    'asphalt': '#0072B2', 'paved': '#56B4E9', 'concrete': '#39568C', 'paving_stones': '#7B3294', 
    'sett': '#B07AA1', 'cobblestone': '#9C755F', 'tiles': '#5177C2', 'gravel': '#E66101', 'rocks': '#51C293', 
    'fine_gravel': '#D28E36', 'compacted': '#8C564B', 'ground': '#6B4423', 'dirt': '#C76D3B', 'sas': '#51B4C2',
    'earth': '#B15928', 'unpaved': '#A6611A', 'sand': '#C5A65A', 'grass': '#4D9221', 'wood': '#AF7AA1', 
    'metal': '#79706E', 'rock': '#6C757D', 'mud': '#5D4037', 'scree': '#AFAFAF', 'grass_paver': '#59A14F', 
    'concrete:lanes': '#4E79A7', 'concrete:plates': '#2F4B7C', 'pebblestone': '#BE7C4D', 'tartan': '#C25156',
    'missing surface tag': '#999999', 'no OSM match': '#111111', 'acrylic': '#C28151', 'driving_plates': '#5169C2', 
    'artificial_turf': '#6051C2', 'bricks': '#64C251', 'chipseal': '#C25185', 'clay': '#51A5C2', 'grus': '#51C2B9',
    'com': '#C2BD51', 'composite': '#9D51C2', 'dirr': '#51C27C', 'dirt/sand': '#C25B51', 'grass_paver;sand': '#89C251', 
    'grave': '#C251AA',  'metal_grate': '#C29851', 'metal_grid': '#7751C2', 'missing': '#999999', 'mixed': '#C2516D', 
    'unhewn_cobblestone': '#98C251', 'non-sealed_chipseal': '#518EC2', 'pakket': '#AFC251', 'plastic': '#B451C2', 
    'rubber': '#C27251', 'rubbercrumb': '#5151C2', 'sand,_kies': '#72C251', 'verdichtetw': '#C251B9', 'stone': '#51C26D', 
    'sand/gravel/stones': '#C25193',  'steel': '#C2AF51', 'woodchips': '#51C2AA', 'stepping_stones': '#8E51C2', 
}

### Extract area of interest from OSM data

In [ ]:
required_osm_columns = {"geometry", "lit", "surface"}

rebuild_osm_cache = True

### Time: ≈ 2 min
if OSM_EDGES_PATH.exists():
    osm_edges = gpd.read_parquet(OSM_EDGES_PATH)

    missing_columns = (required_osm_columns - set(osm_edges.columns))

    if not missing_columns:
        rebuild_osm_cache = False
        print(f"Loaded cached {SAMPLE} OSM bicycle network.")

    else:
        print(f"The cached file is missing: {sorted(missing_columns)}")
        print("Rebuilding the OSM cache...")

if rebuild_osm_cache:
    from pyrosm import OSM  # Only needed when extracting a new cache.
    print(f"Extracting the {SAMPLE} bicycle network from the Denmark PBF...")

    osm_reader = OSM(str(PBF_PATH), bounding_box=bike_loops_bbox_wgs84)

    osm_edges = osm_reader.get_network(network_type="cycling", extra_attributes=["lit", "surface"])

    osm_edges = osm_edges.loc[
        osm_edges.geometry.notna()
        & ~osm_edges.geometry.is_empty
    ].copy()

    osm_edges = osm_edges.explode( index_parts=False, ignore_index=True)
    osm_edges = osm_edges.to_crs(BIKE_LOOPS_CRS)

    columns_to_keep = [
        column
        for column in [
            "id",
            "name",
            "highway",
            "lit",
            "surface",
            "geometry",
        ]
        if column in osm_edges.columns
    ]

    osm_edges = osm_edges[columns_to_keep].copy()
    osm_edges.to_parquet(OSM_EDGES_PATH)
    print(f"Saved {SAMPLE.title()} OSM bicycle network.")

# Cached networks must also use the selected loop network coordinate system.
osm_edges = osm_edges.to_crs(BIKE_LOOPS_CRS)

print(f"Number of OSM bicycle edges: {len(osm_edges):,}")
print("OSM CRS:", osm_edges.crs)

### Surface groups on the entire OSM network of sample

In [ ]:
print("Lit values:")
display(
    osm_edges["lit"]
    .fillna("missing")
    .value_counts()
    .to_frame("edges")
)

# print("Surface values:")
# display(
#     osm_edges["surface"]
#     .fillna("missing")
#     .value_counts()
#     .to_frame("edges")
# )

# Audit every raw OSM tag, including those not selected onto loop links.
observed_surfaces = osm_edges["surface"].map(
    lambda value: "missing surface tag" if pd.isna(value) or not str(value).strip()
    else str(value).strip().lower()
)
unknown_surfaces = set(observed_surfaces) - set(SURFACE_GROUPS)
missing_colours = set(observed_surfaces) - set(surface_palette)
if unknown_surfaces or missing_colours:
    raise ValueError(f"Unassigned groups: {sorted(unknown_surfaces)}; colours: {sorted(missing_colours)}")
surface_assignments = observed_surfaces.value_counts().rename_axis("surface").reset_index(name="osm_edges")
surface_assignments["surface_group"] = surface_assignments["surface"].map(SURFACE_GROUPS)
surface_assignments["colour"] = surface_assignments["surface"].map(surface_palette)
surface_assignments.to_csv(OUTPUT_DIR / "surface_assignments.csv", index=False)
display(surface_assignments)


In [ ]:
# plot_bicycle_networks(
#     sample=SAMPLE,
#     bike_loops_edges=bike_loops_edges,
#     bike_loops_nodes=bike_loops_nodes,
#     bike_loops_bbox_gdf=bike_loops_bbox_gdf,
#     osm_edges=osm_edges,
# )

## Overlap analysis of the two bike networks

In [ ]:
# Shapely supports distance queries with both older and newer GeoPandas.
osm_spatial_index = shapely.STRtree(osm_edges.geometry.to_numpy())
matched_geometries, unmatched_geometries, matched_lengths, osm_candidate_counts = [], [], [], []
matched_osm_ids, matched_lit_values, matched_surface_values = [], [], []
number_of_edges = len(bike_loops_edges)

for position, geometry in enumerate(bike_loops_edges.geometry, start=1):
    # Find only OSM lines within the matching tolerance
    candidate_positions = osm_spatial_index.query(
        geometry,
        predicate="dwithin", # This predicate finds geometries that are within a specified distance of the input geometry.
        distance=TOLERANCE_M,
    )

    candidate_positions = np.unique(candidate_positions)
    candidates = osm_edges.iloc[candidate_positions]

    if candidates.empty:
        matched_geometry = shapely.GeometryCollection()
        unmatched_geometry = geometry

        matched_osm_id = np.nan
        lit_value = np.nan
        surface_value = np.nan

    else:
        # Calculate the complete matched geometry
        local_osm_network = shapely.union_all(candidates.geometry.array)
        local_osm_buffer = local_osm_network.buffer(TOLERANCE_M)
        matched_geometry = geometry.intersection(local_osm_buffer)
        unmatched_geometry = geometry.difference(local_osm_buffer)
        
        # Find which candidate covers the greatest length of this bike-loop edge
        candidate_buffers = (candidates.geometry.buffer(TOLERANCE_M))

        candidate_overlap_lengths = (
            candidate_buffers
            .intersection(geometry)
            .length
        )

        best_position = (
            candidate_overlap_lengths
            .to_numpy()
            .argmax()
        )

        best_candidate = candidates.iloc[best_position]

        matched_osm_id = best_candidate.get("id", np.nan)
        lit_value = best_candidate.get("lit", np.nan)
        surface_value = best_candidate.get("surface", np.nan)

    matched_geometries.append(matched_geometry)
    unmatched_geometries.append(unmatched_geometry)
    matched_lengths.append(matched_geometry.length)
    osm_candidate_counts.append(len(candidates))
    matched_osm_ids.append(matched_osm_id)
    matched_lit_values.append(lit_value)
    matched_surface_values.append(surface_value)

    if position % 500 == 0:
        print(f"Processed {position:,} of {number_of_edges:,} bike-loop edges")

print("Matching completed.")

In [ ]:
bike_loops_overlap = bike_loops_edges.copy()
bike_loops_overlap["bike_loop_length_m"] = bike_loops_overlap.geometry.length
bike_loops_overlap["matched_length_m"] = matched_lengths
bike_loops_overlap["unmatched_length_m"] = bike_loops_overlap["bike_loop_length_m"] - bike_loops_overlap["matched_length_m"]
bike_loops_overlap["coverage_pct"] = 100 * bike_loops_overlap["matched_length_m"] / bike_loops_overlap["bike_loop_length_m"]
bike_loops_overlap["osm_candidate_count"] = osm_candidate_counts
bike_loops_overlap["matched_osm_id"] = matched_osm_ids

bike_loops_overlap["lit"] = matched_lit_values
bike_loops_overlap["lit_group"] = (bike_loops_overlap["lit"].apply(group_lit))
bike_loops_overlap["surface"] = matched_surface_values
bike_loops_overlap["surface_group"] = (bike_loops_overlap["surface"].apply(group_surface))

result_columns = [
    column
    for column in [
        "edge_id",
        "node_start",
        "node_end",
        "bike_loop_length_m",
        "matched_length_m",
        "unmatched_length_m",
        "coverage_pct",
        "osm_candidate_count",
        "matched_osm_id",
        "lit",
        "lit_group",
        "surface",
        "surface_group",
    ]
    if column in bike_loops_overlap.columns
]

matched_edges = gpd.GeoDataFrame(
    bike_loops_overlap[result_columns].copy(),
    geometry=matched_geometries,
    crs=BIKE_LOOPS_CRS,
)

unmatched_edges = gpd.GeoDataFrame(
    bike_loops_overlap[result_columns].copy(),
    geometry=unmatched_geometries,
    crs=BIKE_LOOPS_CRS,
)

matched_edges = matched_edges.loc[~matched_edges.geometry.is_empty].copy()
unmatched_edges = unmatched_edges.loc[~unmatched_edges.geometry.is_empty].copy()

print(f"Matched edges: {len(matched_edges):,}")
print(f"Edges with unmatched geometry: {len(unmatched_edges):,}")

In [ ]:
# overlap_summary = plot_overlap_results(
#     sample=SAMPLE,
#     tolerance_m=TOLERANCE_M,
#     output_dir=OUTPUT_DIR,
#     bike_loops_edges=bike_loops_edges,
#     bike_loops_nodes=bike_loops_nodes,
#     osm_edges=osm_edges,
#     bike_loops_overlap=bike_loops_overlap,
#     matched_edges=matched_edges,
#     unmatched_edges=unmatched_edges,
# )

In [ ]:
# fig, axes = plot_matched_attribute(
#     sample=SAMPLE,
#     osm_edges=osm_edges,
#     matched_edges=matched_edges,
#     unmatched_edges=unmatched_edges,
#     bike_loops_edges=bike_loops_edges,
#     bike_loops_nodes=bike_loops_nodes,
#     tolerance_m=TOLERANCE_M,
#     output_dir=OUTPUT_DIR,
# )

## Interactive surface segments on the loop network
This section displays the **raw OSM surface values** on the loop geometry, rather than giving an entire link just one surface. It is separate from the existing overlap analysis above.

Nearby OSM edges are buffered by `TOLERANCE_M`. Where buffers overlap, the edge with the greatest total overlap with the current loop link takes precedence, matching the existing best-overlap policy. Changes in the winning surface create separate segments; adjacent equal surfaces are merged. These are matched surface assignments within the tolerance, not verified physical surface boundaries. Buffer boundaries are polygonal approximations.

The map distinguishes missing surface tags from stretches with no matching OSM edge. Hover over a segment for its surface, paved/unpaved group, loop edge ID, and length. It uses the existing cached OSM data and does not download data or save additional plots.


Surface groups follow the explicit project assignments in the setup functions cell. Existing paved/unpaved choices are preserved; ambiguous tags are `other`, absent tags are `missing`, and stretches without a nearby OSM match are `unmatched`. Every observed raw tag has an explicit map colour. The audit table is saved to `OUTPUT_DIR / "surface_assignments.csv"`; new unassigned tags stop the analysis for review.


In [ ]:
# Split the loop geometry wherever the assigned OSM surface changes.
from collections import defaultdict
from itertools import groupby
from shapely.ops import substring

surface_geometries = osm_edges.geometry.to_numpy()
surface_tree = shapely.STRtree(surface_geometries)
surface_buffer_cache = {}
surface_records = []

def surface_line_components(geometry):
    if geometry.geom_type == "LineString":
        yield geometry
    elif geometry.geom_type in ("MultiLineString", "GeometryCollection"):
        for component in geometry.geoms:
            yield from surface_line_components(component)

if bike_loops_edges.crs != osm_edges.crs or not bike_loops_edges.crs.is_projected:
    raise ValueError("Surface matching requires the same projected, metre-based CRS.")
if not bike_loops_edges.geom_type.eq("LineString").all() or not bike_loops_edges.is_simple.all():
    raise ValueError("Surface segmentation requires simple LineString loop links.")

for link_id, loop_link in bike_loops_edges.iterrows():
    geometry = loop_link.geometry
    if geometry.length <= 0:
        continue
    candidate_positions = surface_tree.query(geometry, predicate="dwithin", distance=TOLERANCE_M)
    events = [(0.0, None, 0), (float(geometry.length), None, 0)]
    candidates = {}
    for position in candidate_positions:
        position = int(position)
        if position not in surface_buffer_cache:
            surface_buffer_cache[position] = shapely.buffer(
                surface_geometries[position], TOLERANCE_M, quad_segs=16
            )
        overlap = geometry.intersection(surface_buffer_cache[position])
        if overlap.length <= 0:
            continue
        raw_surface = osm_edges.iloc[position]["surface"]
        raw_surface = "missing surface tag" if pd.isna(raw_surface) or not str(raw_surface).strip() else str(raw_surface).strip().lower()
        candidates[position] = {"surface": raw_surface, "overlap_m": overlap.length}
        for component in surface_line_components(overlap):
            start, end = sorted((
                float(geometry.project(shapely.Point(component.coords[0]))),
                float(geometry.project(shapely.Point(component.coords[-1]))),
            ))
            if end > start:
                events.extend([(start, position, 1), (end, position, -1)])
    events.sort(key=lambda event: event[0])
    boundaries = [(distance, list(group)) for distance, group in groupby(events, key=lambda event: event[0])]
    active = defaultdict(int)
    intervals = []
    for i, (start, changes) in enumerate(boundaries[:-1]):
        for _, position, change in changes:
            if position is not None:
                active[position] += change
        end = boundaries[i + 1][0]
        if end <= start:
            continue
        available = [position for position, count in active.items() if count > 0]
        winner = min(available, key=lambda position: (-candidates[position]["overlap_m"], position)) if available else None
        surface = candidates[winner]["surface"] if winner is not None else "no OSM match"
        if intervals and intervals[-1][2] == surface and intervals[-1][1] == start:
            intervals[-1] = (intervals[-1][0], end, surface)
        else:
            intervals.append((start, end, surface))
    for segment_number, (start, end, surface) in enumerate(intervals, start=1):
        surface_records.append({
            "link_id": link_id, "edge_id": loop_link.get("edge_id", link_id),
            "segment": segment_number, "surface": surface,
            "surface_group": "unmatched" if surface == "no OSM match" else group_surface(surface),
            "from_m": start, "to_m": end, "length_m": end - start,
            "geometry": substring(geometry, start, end),
        })

bike_loops_surface_segments = gpd.GeoDataFrame(surface_records, geometry="geometry", crs=bike_loops_edges.crs)
surface_length_difference = (
    bike_loops_surface_segments.groupby("link_id")["length_m"].sum()
    - bike_loops_edges.geometry.length
)
if not np.allclose(surface_length_difference.to_numpy(), 0, atol=1e-6, rtol=0):
    raise AssertionError("Surface segments do not preserve the loop-link lengths.")

print(f"{len(bike_loops_surface_segments):,} surface segments on {len(bike_loops_edges):,} loop links")

In [ ]:
# Surface distribution by loop-network length, including missing/unmatched segments.
surface_total_length_m = bike_loops_surface_segments["length_m"].sum()
if surface_total_length_m <= 0:
    raise ValueError("There is no loop-network length to summarise.")

surface_distribution = (
    bike_loops_surface_segments.groupby("surface", dropna=False)
    .agg(segments=("surface", "size"), length_m=("length_m", "sum"))
    .assign(
        length_km=lambda table: table["length_m"] / 1000,
        length_pct=lambda table: 100 * table["length_m"] / surface_total_length_m,
    )
    .sort_values("length_pct", ascending=False)
    [["segments", "length_km", "length_pct"]]
)
surface_distribution.index.name = "Surface"
print(f"Surface percentages of total loop length ({surface_total_length_m / 1000:,.2f} km):")
display(surface_distribution.round(2))
print(f"Total: {surface_distribution['length_pct'].sum():.2f}% (includes missing surface tags and no OSM match)")

In [ ]:
# Interactive map of every matched surface segment, using raw surface tags.
import folium
surface_categories = sorted(bike_loops_surface_segments["surface"].unique())

surface_map = bike_loops_surface_segments.explore(
    column="surface", categorical=True, categories=surface_categories,
    cmap=[surface_palette[surface] for surface in surface_categories],
    tiles=None, style_kwds={"weight": 4, "opacity": 1},
    highlight_kwds={"weight": 4},
    tooltip=["edge_id", "surface", "surface_group", "length_m"],
    tooltip_kwds={"aliases": ["Loop edge", "OSM surface", "Surface group", "Segment length (m)"], "localize": True},
)

# Embed map images so VS Code does not need to fetch background tiles.
import hashlib
import json
import requests

def add_embedded_background(service, name, attribution, show):
    x0, y0, x1, y1 = bike_loops_edges.to_crs(3857).total_bounds
    padding = max(x1 - x0, y1 - y0) * 0.07
    image_bbox = [float(x0 - padding), float(y0 - padding), float(x1 + padding), float(y1 + padding)]
    cache_key = hashlib.sha256(json.dumps([service, image_bbox, [1200, 1400]]).encode()).hexdigest()[:16]
    cache_dir = Path(OUTPUT_DIR) / "basemap_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    image_path = cache_dir / f"{service}_{cache_key}.png"
    metadata_path = image_path.with_suffix(".json")

    if image_path.exists() and metadata_path.exists():
        metadata = json.loads(metadata_path.read_text())
    else:
        print(f"Downloading {name.lower()} once...", flush=True)
        response = requests.get(
            f"https://server.arcgisonline.com/ArcGIS/rest/services/{service}/MapServer/export",
            params={
                "bbox": ",".join(map(str, image_bbox)), "bboxSR": 3857,
                "imageSR": 3857, "size": "1200,1400", "format": "png32", "f": "json",
            }, timeout=60,
        )
        response.raise_for_status()
        result = response.json()
        if "error" in result or "href" not in result:
            raise RuntimeError(f"Could not download {name}: {result.get('error', result)}")
        image_response = requests.get(result["href"], timeout=60)
        image_response.raise_for_status()
        if not image_response.headers.get("Content-Type", "").startswith("image/"):
            raise RuntimeError(f"{name} returned no map image.")
        image_path.write_bytes(image_response.content)
        # The export server may adjust the requested extent to the image aspect ratio.
        metadata = {"extent": result["extent"]}
        metadata_path.write_text(json.dumps(metadata))
    extent = metadata["extent"]
    west, south, east, north = gpd.GeoSeries(
        [shapely.box(extent["xmin"], extent["ymin"], extent["xmax"], extent["ymax"])],
        crs=3857,
    ).to_crs(4326).total_bounds
    folium.raster_layers.ImageOverlay(
        image=str(image_path), bounds=[[south, west], [north, east]],
        name=name, attribution=attribution, overlay=False, control=True,
        show=show, opacity=1, interactive=False,
    ).add_to(surface_map)


add_embedded_background(
    "World_Topo_Map", "Topographic background",
    "Tiles &copy; Esri; sources: Esri, USGS, NOAA and contributors", True,
)

add_embedded_background(
    "World_Imagery", "Satellite background",
    "Tiles &copy; Esri; imagery: Esri, Maxar, Earthstar Geographics and contributors", False,
)

folium.LayerControl(position="topright", collapsed=False).add_to(surface_map)
# These cached images cover the selected region at a fixed resolution; detail will soften when zoomed in.
display(surface_map)

In [ ]:
import hashlib
import json
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from shapely.geometry import box

BACKGROUND = "World_Topo_Map"  # or "World_Imagery"
MAP_PADDING = 0.015  # Tight crop around the network, without the former 7% margin.

segments = bike_loops_surface_segments.to_crs(3857)

# Find the same cached background as the interactive map.
x0, y0, x1, y1 = bike_loops_edges.to_crs(3857).total_bounds
cache_padding = max(x1 - x0, y1 - y0) * 0.07
image_bbox = [
    float(x0 - cache_padding), float(y0 - cache_padding),
    float(x1 + cache_padding), float(y1 + cache_padding),
]
cache_key = hashlib.sha256(
    json.dumps([BACKGROUND, image_bbox, [1200, 1400]]).encode()
).hexdigest()[:16]
image_path = Path(OUTPUT_DIR) / "basemap_cache" / f"{BACKGROUND}_{cache_key}.png"
metadata_path = image_path.with_suffix(".json")
if not image_path.exists() or not metadata_path.exists():
    raise FileNotFoundError("Run the interactive-map cell first to cache the background.")
extent = json.loads(metadata_path.read_text())["extent"]
background_image = plt.imread(image_path)
background_extent = [extent["xmin"], extent["xmax"], extent["ymin"], extent["ymax"]]

# Separate Bornholm geographically; only its display position changes.
bornholm_area = gpd.GeoSeries([box(14.5, 54.9, 15.5, 55.5)], crs=4326).to_crs(3857).iloc[0]
is_bornholm = segments.geometry.intersects(bornholm_area)
bornholm_segments = segments.loc[is_bornholm]
mainland_segments = segments.loc[~is_bornholm]
if bornholm_segments.empty or mainland_segments.empty:
    raise ValueError("This layout requires both Bornholm and the rest of Denmark.")

fig, ax = plt.subplots(figsize=(12, 14), facecolor="white")
fig.subplots_adjust(left=0.005, right=0.79, bottom=0.015, top=0.96)


def draw_surface_panel(panel, data, padding_fraction, image=background_image, image_extent=background_extent):
    panel.imshow(image, extent=image_extent, origin="upper", zorder=0)
    data.plot(
        ax=panel, color=data["surface"].map(surface_palette),
        linewidth=1, zorder=2,
    )
    xmin, ymin, xmax, ymax = data.total_bounds
    margin = max(xmax - xmin, ymax - ymin) * padding_fraction
    panel.set_xlim(xmin - margin, xmax + margin)
    panel.set_ylim(ymin - margin, ymax + margin)
    panel.set_aspect("equal")
    panel.set_axis_off()


draw_surface_panel(ax, mainland_segments, MAP_PADDING)

# Centre Bornholm at (0.75, 0.75): the centre of the upper-right quadrant.
bornholm_ax = ax.inset_axes([0.61, 0.61, 0.28, 0.28], zorder=5)
# Prefer an existing Bornholm background to avoid enlarging the low-resolution national image.
inset_image, inset_extent = background_image, background_extent
bx0, by0, bx1, by1 = bornholm_segments.total_bounds
candidates = []
for metadata_file in (Path(OUTPUT_DIR).parent / "bornholm/basemap_cache").glob(f"{BACKGROUND}_*.json"):
    image_file = metadata_file.with_suffix(".png")
    e = json.loads(metadata_file.read_text())["extent"]
    if image_file.exists() and e["xmin"] <= bx0 and e["xmax"] >= bx1 and e["ymin"] <= by0 and e["ymax"] >= by1:
        candidates.append(((e["xmax"] - e["xmin"]) * (e["ymax"] - e["ymin"]), image_file, e))
if candidates:
    _, inset_file, e = min(candidates, key=lambda item: item[0])
    inset_image = plt.imread(inset_file)
    inset_extent = [e["xmin"], e["xmax"], e["ymin"], e["ymax"]]
draw_surface_panel(bornholm_ax, bornholm_segments, 0.04, inset_image, inset_extent)
bornholm_ax.set_title("Bornholm (inset)", fontsize=11, pad=5)

# Use original segment lengths in metres, not Web Mercator display lengths.
# All segments contribute, including Bornholm, missing tags and unmatched stretches.
surface_lengths = bike_loops_surface_segments.groupby("surface")["length_m"].sum()
if surface_lengths.sum() <= 0:
    raise ValueError("There is no positive network length to summarise.")
surface_percentages = 100 * surface_lengths / surface_lengths.sum()
categories = sorted(segments["surface"].unique())
legend_labels = {}
for surface in categories:
    percentage = surface_percentages.loc[surface]
    percentage_text = "<0.1" if 0 < percentage < 0.1 else f"{percentage:.1f}".rstrip("0").rstrip(".")
    legend_labels[surface] = f"{surface} ({percentage_text}%)"
ax.legend(
    handles=[Patch(facecolor=surface_palette[surface], label=legend_labels[surface]) for surface in categories],
    title="Surface type", loc="upper left", bbox_to_anchor=(1.01, 1),
    borderaxespad=0, frameon=False, fontsize=9, title_fontsize=10,
)
ax.set_title("Denmark: loop-network surfaces", pad=12)

attribution = (
    "Esri, Maxar, Earthstar Geographics" if BACKGROUND == "World_Imagery"
    else "Esri, USGS, NOAA and contributors"
)
ax.text(
    0.01, 0.01, f"Background: {attribution}", transform=ax.transAxes,
    fontsize=7, bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
)

output_path = Path(OUTPUT_DIR) / f"{SAMPLE}_surface_map.png"
fig.savefig(output_path, dpi=300, bbox_inches="tight", pad_inches=0.02)
plt.show()
print("Saved:", output_path)


## Try imputing short surface segments
Set `MAX_IMPUTE_LENGTH_M` below. Within each loop link, only a segment labelled `missing surface tag` or `no OSM match` (or with an empty tag), shorter than this threshold, takes its neighbours' surface **when both neighbours agree on a known surface**. Existing surface tags are never replaced. The comparison uses the raw `surface` labels, not just paved/unpaved groups. End segments are unchanged.

This is one simultaneous pass using the original labels, not an iterative smoothing operation. The original `bike_loops_surface_segments` stays unchanged. The new `bike_loops_surface_segments_imputed` retains each segment's geometry and length, with `surface_original` and `surface_imputed` for auditing. Segments are not merged, so the before/after rows remain directly comparable. The second map uses the same background layers and colours.


In [ ]:
MAX_IMPUTE_LENGTH_M = 100.0  # X: change this threshold and rerun this cell

def impute_short_surface_segments(segments, max_length_m):
    if not np.isfinite(max_length_m) or max_length_m < 0:
        raise ValueError("The imputation threshold must be a finite, nonnegative number of metres.")
    result = segments.sort_values(["link_id", "from_m"], kind="stable").copy()
    result["surface_original"] = result["surface"]
    original_by_link = result.groupby("link_id", sort=False)["surface_original"]
    previous_surface = original_by_link.shift(1)
    next_surface = original_by_link.shift(-1)
    unknown_surfaces = ["missing surface tag", "no OSM match", ""]
    missing_surface = (
        result["surface_original"].isna()
        | result["surface_original"].isin(unknown_surfaces)
    )
    result["surface_imputed"] = (
        missing_surface
        & (result["length_m"] < max_length_m)
        & ~previous_surface.isin(unknown_surfaces)
        & previous_surface.notna()
        & next_surface.notna()
        & previous_surface.eq(next_surface)
        & result["surface_original"].ne(previous_surface)
    ).fillna(False)
    changed = result["surface_imputed"]
    result.loc[changed, "surface"] = previous_surface.loc[changed]
    result.loc[changed, "surface_group"] = result.loc[changed, "surface"].map(
        lambda surface: "unmatched" if surface == "no OSM match" else group_surface(surface)
    )
    return result


bike_loops_surface_segments_imputed = impute_short_surface_segments(
    bike_loops_surface_segments, MAX_IMPUTE_LENGTH_M
)
imputed_segments = bike_loops_surface_segments_imputed.loc[
    bike_loops_surface_segments_imputed["surface_imputed"]
]
print(
    f"Changed {len(imputed_segments):,} segments shorter than {MAX_IMPUTE_LENGTH_M:g} m "
    f"({imputed_segments['length_m'].sum():,.2f} m in total)."
)
display(imputed_segments[["link_id", "edge_id", "segment", "surface_original", "surface", "length_m"]])

# Build a fresh map: copying a rendered map can retain old GeoJSON scripts.
# Reuse only its background layers so no background images are downloaded again.
from copy import deepcopy
imputed_surface_map = bike_loops_surface_segments_imputed.explore(
    column="surface", categorical=True, categories=surface_categories,
    cmap=[surface_palette[surface] for surface in surface_categories],
    tiles=None,
    style_kwds={"weight": 3, "opacity": 1}, highlight_kwds={"weight": 6},
    tooltip=["edge_id", "surface_original", "surface", "surface_imputed", "length_m"],
    tooltip_kwds={
        "aliases": ["Loop edge", "Original surface", "Imputed surface", "Changed", "Segment length (m)"],
        "localize": True,
    },
)
for layer in surface_map._children.values():
    if isinstance(layer, (folium.raster_layers.ImageOverlay, folium.raster_layers.TileLayer)):
        deepcopy(layer).add_to(imputed_surface_map)
folium.LayerControl(position="topright", collapsed=False).add_to(imputed_surface_map)
display(imputed_surface_map)
